# Gaussian Mixture EM Practice

**Topic:** Generative Modeling &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `X` `(2400, 2)`: unlabeled points from a 3-component Gaussian mixture with different weights, tilted elongated covariances and some overlap in the tails. Plot it!
- The true component of each point is hidden in the evaluation cell (scored with Adjusted Rand Index, so component numbering doesn't matter).

## Your task

1. Implement EM for a 3-component full-covariance GMM in NumPy (log-space or with small regularization `+1e-6·I` on covariances for stability). ~100 iterations from a few random starts is plenty. (You may fit `sklearn.mixture.GaussianMixture` to compare with your own result, but the `predictions` array you submit must come from the EM code you write.)
2. Assign each point to its highest-responsibility component.
3. Set `predictions` to the `(2400,)` array of component labels.

## Evaluation

Adjusted Rand Index vs the hidden truth must be ≥ 0.70. The check scores your labels only. Fitting `sklearn.mixture.GaussianMixture` and handing over its labels passes the gate without doing the exercise.


In [ ]:
import numpy as np

rng = np.random.default_rng(561)

def rot(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

_means = np.array([[0.0, 0.0], [4.5, 1.5], [1.5, 4.8]])
_covs = [
    rot(0.5) @ np.diag([2.8, 0.25]) @ rot(0.5).T,
    rot(-0.7) @ np.diag([2.2, 0.3]) @ rot(-0.7).T,
    rot(1.2) @ np.diag([1.8, 0.35]) @ rot(1.2).T,
]
_weights = np.array([0.45, 0.35, 0.20])

_n = 2400
_comp = rng.choice(3, size=_n, p=_weights)
X = np.vstack([rng.multivariate_normal(_means[c], _covs[c]) for c in _comp])
del _comp  # rebuilt from the same seed in the evaluation cell

print(f"X {X.shape}: three tilted, overlapping Gaussians.")
print("# To plot it: import matplotlib.pyplot as plt; plt.scatter(X[:,0], X[:,1], s=3)")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: Adjusted Rand Index against the hidden components
# =====================================================================
import numpy as np
from sklearn.metrics import adjusted_rand_score

# The hidden truth, rebuilt from the same seed as the data cell.
_truth = np.random.default_rng(561).choice(3, size=2400, p=np.array([0.45, 0.35, 0.20]))

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
if _pred.shape != _truth.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_truth.shape}.")

_ari = float(adjusted_rand_score(_truth, _pred))
print("=" * 52)
print(f"Adjusted Rand Index : {_ari:.4f}   (target: >= 0.70)")
_counts = np.unique(_pred.astype(int), return_counts=True)
print(f"Your component sizes: {dict(zip(*_counts))}")
print("=" * 52)
print("PASSED: the three components were recovered!" if _ari >= 0.70 else "NOT PASSED yet: try more random restarts and check the covariance update.")


## Solution

Full EM: responsibilities from log-space Gaussian densities (E-step), then weighted means, covariances (+1e-6 I for stability), and mixing weights (M-step). Keep the best of a few random restarts by log-likelihood and assign each point to its top component.

```python
import numpy as np

def log_gauss(X, mean, cov):
    d = X.shape[1]
    diff = X - mean
    inv = np.linalg.inv(cov)
    _, logdet = np.linalg.slogdet(cov)
    return -0.5 * (d * np.log(2 * np.pi) + logdet + np.einsum("ni,ij,nj->n", diff, inv, diff))

def em(X, k, seed, iters=100):
    r = np.random.default_rng(seed)
    n, d = X.shape
    means = X[r.choice(n, k, replace=False)]
    covs = np.array([np.cov(X.T) for _ in range(k)])
    weights = np.full(k, 1.0 / k)
    ll_prev = -np.inf
    for _ in range(iters):
        # E-step in log space
        log_r = np.stack([np.log(weights[j]) + log_gauss(X, means[j], covs[j]) for j in range(k)], axis=1)
        norm = log_r.max(axis=1, keepdims=True)
        resp = np.exp(log_r - norm)
        ll = float((norm.ravel() + np.log(resp.sum(axis=1))).sum())
        resp /= resp.sum(axis=1, keepdims=True)
        # M-step
        nk = resp.sum(axis=0)
        weights = nk / n
        means = (resp.T @ X) / nk[:, None]
        for j in range(k):
            diff = X - means[j]
            covs[j] = (resp[:, j][:, None] * diff).T @ diff / nk[j] + 1e-6 * np.eye(d)
        if abs(ll - ll_prev) < 1e-7 * max(1.0, abs(ll)):
            break
        ll_prev = ll
    return ll, resp.argmax(axis=1)

best_ll, best_labels = -np.inf, None
for seed in range(5):
    ll, labels = em(X, 3, seed)
    if ll > best_ll:
        best_ll, best_labels = ll, labels

print(f"best log-likelihood: {best_ll:.1f}")
predictions = best_labels
```
